# 04 - Poison-fraction sweep (pre-registered) + optional all-labels runs
Pre-registration: `experiments/prereg_sweep.md` (committed before any run). This notebook:
1. copies the prepared Stage 1/1b files into `/kaggle/working/nids_out` and checks that their sha256 equal the ones `main` used;
2. runs `repro_s42` (the `main` settings again) and compares its **validation decisions** with `main`;
3. runs the 12 sweep runs `sweep_f{0.75,0.9,0.95,1.0}_s{42,43,44}`;
4. makes the **validation-only** decision table (`s6_sweep.py --decisions`) → **STOP**: send `decisions.csv`, it is committed to git;
5. only then the test table (`s6_sweep.py --test`);
6. (optional, off) part B: the all-labels schedule, CII then CI, seed 42;
7. zips only the new outputs.

`main` is never touched: it is read from the read-only results dataset, and `s3_continual.py` refuses to overwrite any completed run.

**Inputs (Add Input):**
| input | what it provides |
|---|---|
| code dataset `dvdhere/nids-code` | the cleaned parquet splits (`results_01`, found by searching for `clean_train.parquet`) and, after a new dataset version, the code |
| results dataset `dvdhere/nids-results` | the completed `main` run (`runs/cii/main`), for the reproduction check |
| (or) Internet ON + `git clone` | the newest code from GitHub (cell 1 prefers it) |

Accelerator: **GPU T4**.

In [1]:
# 1. Locate code, prepared files and main; copy prepared files to a writable folder
import glob, hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
# the marker s6_sweep.py exists only in the new code, so an old code dataset is never picked by mistake
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s6_sweep.py")] +
     sorted(glob.glob("/kaggle/input/**/s6_sweep.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "new code not found: turn Internet ON (git clone) or attach a code dataset version with s6_sweep.py"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
from common import load_json

git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git or "(not a git checkout) s3 sha256 " +
      hashlib.sha256(Path(CODE_DIR, "s3_continual.py").read_bytes()).hexdigest()[:16])

WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
PREP = X.find_dir("clean_train.parquet", "NIDS_PREPARED")   # results_01 inside the code dataset
assert PREP, "clean_train.parquet not found under /kaggle/input: attach the code dataset that contains results_01"
if PREP.resolve() != WORK.resolve():
    shutil.copytree(PREP, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns("runs"))
MAIN_OUT = X.find_dir("runs/cii/main/run_manifest.json", "NIDS_RESULTS")
MAIN = MAIN_OUT / "runs" / "cii" / "main" if MAIN_OUT else None
print("prepared:", PREP, "->", WORK, "\nmain run:", MAIN)

# the copied files must be exactly the files main was built from
if MAIN:
    want = load_json(MAIN / "config.json")["inputs"]
    def sha(p):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
        return h.hexdigest()
    bad = {n: "missing" if not (WORK / n).exists() else "different" for n in want
           if not (WORK / n).exists() or sha(WORK / n) != want[n]}
    assert not bad, f"prepared files differ from main's inputs: {bad}"
    print(f"all {len(want)} Stage 1/1b inputs match main's sha256")
else:
    print("WARNING: main not found - the reproduction check (H0) will be skipped")

def run(script, *args):
    cmd = [sys.executable, str(Path(CODE_DIR) / script), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

Cloning into '/kaggle/working/code'...


code: /kaggle/working/code | git commit: 3e7807c1000c48b6af5d615f8b6d4d45b82b962a
prepared: /kaggle/input/datasets/dvdhere/nids-code/nids/notebooks/results_01/nids_out -> /kaggle/working/nids_out 
main run: /kaggle/input/datasets/dvdhere/nids-results/nids_out/runs/cii/main
all 9 Stage 1/1b inputs match main's sha256


In [2]:
from pathlib import Path
w = Path("/kaggle/working/nids_out")
runs = list((w / "runs" / "cii").glob("sweep_*/run_manifest.json"))
repro = (w / "runs" / "cii" / "repro_s42" / "run_manifest.json").is_file()
decisions = (w / "sweep" / "decisions.csv").is_file()
print("sweep runs:", len(runs), "repro:", repro, "decisions:", decisions)

sweep runs: 0 repro: False decisions: False


## 2. GPU and dry run (loads everything, checks it against Stage 1b, trains nothing)

In [3]:
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)
SCENARIO = "cii"
run("s3_continual.py", "--scenario", SCENARIO, "--dry-run")

Tesla T4 2.11.0+cu128
$ /kaggle/working/code/s3_continual.py --scenario cii --dry-run
scenario=cii schedule=[['Syn'], ['LDAP'], ['UDP']] device=cuda features=77 val_rows=23263 test_rows=46527
train rows per experience: {'E0': {'BENIGN': 19055, 'Syn': 13890}, 'E1': {'BENIGN': 19054, 'LDAP': 13890}, 'E2': {'BENIGN': 19054, 'UDP': 13890}}
val rows: {'BENIGN': 9507, 'Syn': 5973, 'LDAP': 2296, 'UDP': 5487} 
test rows: {'BENIGN': 19028, 'Syn': 11946, 'LDAP': 4576, 'UDP': 10977}
protected rows excluded: 166 | inputs match Stage 1b: OK
dry run: everything loads and matches Stage 1b; nothing trained or written


## 3. Reproduction: `repro_s42` = the `main` settings (defaults, seed 42, f = 0.5)
H0: same gate decisions as `main`. On GPU exact equality of every number is not guaranteed; the max difference is printed, not hidden.

In [4]:
RUNS = WORK / "runs" / SCENARIO
def done(name): return (RUNS / name / "run_manifest.json").exists()
t0 = time.time()
if not done("repro_s42"):
    run("s3_continual.py", "--scenario", SCENARIO, "--run", "repro_s42", "--resume")
run("s4_report.py", "--scenario", SCENARIO, "--run", "repro_s42")
print(f"repro: {time.time() - t0:.0f} s")
if MAIN:
    rows = []
    for arm in ("clean", "poisoned", "poisoned_repair"):
        for m, r in zip(load_json(MAIN / arm / "rounds.json"), load_json(RUNS / "repro_s42" / arm / "rounds.json")):
            rows.append({"arm": arm, "E": m["experience"], "attempt": (m["attempt"], r["attempt"]),
                         "accepted main/repro": (m["accepted"], r["accepted"]),
                         "max |diff| val recall": max(abs(m["val"]["recall"][k] - r["val"]["recall"][k]) for k in m["val"]["recall"])})
    print(pd.DataFrame(rows).to_string(index=False))

$ /kaggle/working/code/s3_continual.py --scenario cii --run repro_s42 --resume
scenario=cii schedule=[['Syn'], ['LDAP'], ['UDP']] device=cuda features=77 val_rows=23263 test_rows=46527
train rows per experience: {'E0': {'BENIGN': 19055, 'Syn': 13890}, 'E1': {'BENIGN': 19054, 'LDAP': 13890}, 'E2': {'BENIGN': 19054, 'UDP': 13890}}
val rows: {'BENIGN': 9507, 'Syn': 5973, 'LDAP': 2296, 'UDP': 5487} 
test rows: {'BENIGN': 19028, 'Syn': 11946, 'LDAP': 4576, 'UDP': 10977}
protected rows excluded: 166 | inputs match Stage 1b: OK
E0 trained in 21s | val recall {'BENIGN': 1.0, 'Syn': 0.995312238406161} | benign FAR 0.000
[clean] E1 C1: ACCEPT []
[clean] E2 C1: ACCEPT []
[poisoned] E1 C1: ACCEPT []
[poisoned] E2 C1: ACCEPT []
[poisoned_repair] E1 C1: ACCEPT []
[poisoned_repair] E2 C1: ACCEPT []
done. teacher unchanged (sha256 9ad67b4de9a8). outputs in /kaggle/working/nids_out/runs/cii/repro_s42
$ /kaggle/working/code/s4_report.py --scenario cii --run repro_s42
            arm experience attempt  

## 4. Sweep: 4 poison fractions x 3 seeds (settings fixed by the pre-registration)
Each run trains T/A0 for its seed, then the 3 arms; `--forced-r1` adds **diagnostic** R1s on the clean and poisoned arms (logged in `r1_diag.json`, never used for a decision). If the session dies, rerun this cell: finished runs are skipped and unfinished ones resume.

In [5]:
FRACS = [0.75, 0.9, 0.95, 1.0]
SEEDS = [42, 43, 44]
FORCED = "soft,threeway,threeway+relabel"
for f in FRACS:
    for s in SEEDS:
        name = f"sweep_f{f}_s{s}"
        if done(name):
            print("done:", name); continue
        t0 = time.time()
        run("s3_continual.py", "--scenario", SCENARIO, "--run", name, "--seed", s, "--poison-frac", f,
            "--forced-r1", FORCED, "--resume")
        run("s4_report.py", "--scenario", SCENARIO, "--run", name)
        print(f"{name}: {time.time() - t0:.0f} s", flush=True)

$ /kaggle/working/code/s3_continual.py --scenario cii --run sweep_f0.75_s42 --seed 42 --poison-frac 0.75 --forced-r1 soft,threeway,threeway+relabel --resume
scenario=cii schedule=[['Syn'], ['LDAP'], ['UDP']] device=cuda features=77 val_rows=23263 test_rows=46527
train rows per experience: {'E0': {'BENIGN': 19055, 'Syn': 13890}, 'E1': {'BENIGN': 19054, 'LDAP': 13890}, 'E2': {'BENIGN': 19054, 'UDP': 13890}}
val rows: {'BENIGN': 9507, 'Syn': 5973, 'LDAP': 2296, 'UDP': 5487} 
test rows: {'BENIGN': 19028, 'Syn': 11946, 'LDAP': 4576, 'UDP': 10977}
protected rows excluded: 166 | inputs match Stage 1b: OK
E0 trained in 18s | val recall {'BENIGN': 1.0, 'Syn': 0.995312238406161} | benign FAR 0.000
[clean] E1 C1: ACCEPT []
[clean] E1 R1_diag[soft] (diagnostic, not a decision): would pass []
[clean] E1 R1_diag[threeway] (diagnostic, not a decision): would pass []
[clean] E1 R1_diag[threeway+relabel] (diagnostic, not a decision): would pass []
[clean] E2 C1: ACCEPT []
[clean] E2 R1_diag[soft] (diag

## 5. Validation-only decision table (no test numbers are read here)

In [6]:
run("s6_sweep.py", "--scenario", SCENARIO, *(["--main-dir", MAIN] if MAIN else []), "--decisions")
display(pd.read_csv(WORK / "sweep" / "hypotheses.csv"))

$ /kaggle/working/code/s6_sweep.py --scenario cii --main-dir /kaggle/input/datasets/dvdhere/nids-results/nids_out/runs/cii/main --decisions
repro vs main: same decisions = True; max |diff| Syn val recall = 0.0
            run             arm  experience attempt  accepted  Syn_val_recall  Syn_floor  Syn_val_mean_p  benign_far
sweep_f0.75_s42           clean           1      C1      True        0.995312   0.945312        0.991445    0.000841
sweep_f0.75_s42           clean           2      C1      True        0.996149   0.945312        0.995267    0.001788
sweep_f0.75_s42        poisoned           1      C1      True        0.994475   0.945312        0.596796    0.001052
sweep_f0.75_s42        poisoned           2      C1      True        0.995145   0.944475        0.721671    0.000841
sweep_f0.75_s42 poisoned_repair           1      C1      True        0.994475   0.945312        0.596796    0.001052
sweep_f0.75_s42 poisoned_repair           2      C1      True        0.995145   0.944475

,hypothesis,claim,evidence,result
0,H0,repro_s42 makes the same gate decisions as main,1/1,supported
1,H1,f=0.75: poisoned C1 at E1 accepted (every seed),2/3,NOT supported
2,H1,f=0.9: poisoned C1 at E1 accepted (every seed),1/3,NOT supported
3,H2,f=0.95: poisoned C1 at E1 rejected (>= 2 of 3 ...,3/3,supported
4,H2,f=1.0: poisoned C1 at E1 rejected (>= 2 of 3 s...,3/3,supported
5,H3,"when C1 was rejected, soft R1 passed the gate ...",11/17,descriptive
6,H4,forced soft R1: mean weight < 0.05 on flipped ...,24/24,supported
7,H5,"f=1.0, forced soft R1 would pass the gate (dia...",0/3,descriptive
8,H5,"f=1.0, forced threeway+relabel R1 would pass t...",3/3,descriptive
9,H6,clean C1 accepted at every update,24/24,supported


## STOP - send `nids_out/sweep/decisions.csv`, `diagnostics.csv` and `hypotheses.csv`
They are committed to git **before** any test number is produced (pre-registration rule 2). After the commit, set `DECISIONS_COMMITTED = True` below and run the next cell.

In [7]:
DECISIONS_COMMITTED = True
if DECISIONS_COMMITTED:
    run("s6_sweep.py", "--scenario", SCENARIO, "--test")
else:
    print("skipped: commit decisions.csv first, then set DECISIONS_COMMITTED = True")

$ /kaggle/working/code/s6_sweep.py --scenario cii --test
 poison_frac             arm  n_seeds  recall_BENIGN_mean  recall_BENIGN_min  recall_BENIGN_max  recall_Syn_mean  recall_Syn_min  recall_Syn_max  recall_LDAP_mean  recall_LDAP_min  recall_LDAP_max  recall_UDP_mean  recall_UDP_min  recall_UDP_max  benign_far_mean  benign_far_min  benign_far_max
        0.75           clean        3            0.998931           0.998161           0.999474         0.995396        0.995396        0.995396          0.985140         0.983173         0.988636         0.998755        0.997723        0.999362         0.001069        0.000526        0.001839
        0.75        poisoned        3            0.999334           0.998686           1.000000         0.995229        0.994977        0.995396          0.656833         0.000000         0.987544         0.666059        0.000000        0.999271         0.000666        0.000000        0.001314
        0.75 poisoned_repair        3            0.999001 

## Part B (optional, OFF) - all-labels schedule, CII then CI, seed 42
Prepared, **not run** yet: nothing here is a result until this part has run.

Rule (`make_all_labels_schedule.py`): an attack class is included if it has >= 1,000 train and >= 200 val rows. That gives **12 of 14 labels**: Portmap (363 train / 60 val) and WebDDoS (132 / 22) are excluded. E0 = BENIGN + Syn (as in `main`); the other 10 classes, sorted by name, arrive two per experience, so there are 6 experiences. Attack rows per class = the smallest included train count (NetBIOS, 9,312). Benign: CII gets 6 disjoint portions (9,528 + 5 x 9,527); CI puts all 57,163 train benign rows in E0.

New Stage 1b files go to `nids_out/all12/` (`--tag all12`); the files `main` and the sweep used are never touched.

**Protocol deviations to report:** the scaler is refit on the new E0 (a different benign portion), so its protected exclusions and the scored rows differ from `main` (no row-for-row comparison); benign is about 34% of each CII experience instead of 58%; one seed only. A refusal by s1b (more than 5% of a class excluded, or fewer than 5,000 benign val/test rows) stops this part; that is a finding to report, not a reason to loosen the check.

In [8]:
RUN_ALL_LABELS = True
TAG = "all12"
if RUN_ALL_LABELS:
    sched = WORK / f"{TAG}_schedule.json"
    run("make_all_labels_schedule.py", "--split-manifest", WORK / "split_manifest.json", "--write", sched)
    cap = load_json(sched)["attack_cap"]
    for sc in ("cii", "ci"):
        run("s1b_experiences.py", "--schedule", sched, "--scenario", sc, "--tag", TAG, "--attack-cap", cap)
    import numpy as np
    tr = pd.read_parquet(WORK / "clean_train.parquet", columns=["_rid", "label"]).set_index("_rid")["label"]
    z = {sc: np.load(WORK / TAG / sc / "experience_rids.npz") for sc in ("cii", "ci")}
    for t in range(len(load_json(sched)["experiences"])):
        a = {sc: set(v for v in z[sc][f"e{t}"] if tr[v] != "BENIGN") for sc in z}
        assert a["cii"] == a["ci"], f"E{t}: CI and CII attack rows differ (protocol deviation)"
    b = [set(v for v in z["cii"][f"e{t}"] if tr[v] == "BENIGN") for t in range(len(z["cii"].files))]
    assert all(not (b[i] & b[j]) for i in range(len(b)) for j in range(i + 1, len(b))), "CII benign overlap"
    print("CI and CII share every attack row; CII benign portions are disjoint")
    for sc in ("cii", "ci"):
        name = f"{TAG}_{sc}_s42"
        if not (WORK / "runs" / sc / name / "run_manifest.json").exists():
            run("s3_continual.py", "--scenario", sc, "--tag", TAG, "--run", name, "--forced-r1", FORCED, "--resume")
        run("s4_report.py", "--scenario", sc, "--run", name)
else:
    print("part B skipped (RUN_ALL_LABELS = False)")

$ /kaggle/working/code/make_all_labels_schedule.py --split-manifest /kaggle/working/nids_out/split_manifest.json --write /kaggle/working/nids_out/all12_schedule.json
{
 "experiences": [
  [
   "Syn"
  ],
  [
   "DNS",
   "LDAP"
  ],
  [
   "MSSQL",
   "NTP"
  ],
  [
   "NetBIOS",
   "SNMP"
  ],
  [
   "SSDP",
   "TFTP"
  ],
  [
   "UDP",
   "UDPLag"
  ]
 ],
 "attack_cap": 9312,
 "labels_used": 12,
 "labels_total": 14,
 "excluded": {
  "Portmap": {
   "train": 363,
   "val": 60
  },
  "WebDDoS": {
   "train": 132,
   "val": 22
  }
 },
 "rule": {
  "min_train": 1000,
  "min_val": 200,
  "e0": "Syn",
  "per_experience": 2
 },
 "cii_benign_per_experience": [
  9528,
  9527,
  9527,
  9527,
  9527,
  9527
 ],
 "ci_benign": [
  57163,
  0,
  0,
  0,
  0,
  0
 ]
}
written /kaggle/working/nids_out/all12_schedule.json; run s1b with --attack-cap 9312
$ /kaggle/working/code/s1b_experiences.py --schedule /kaggle/working/nids_out/all12_schedule.json --scenario cii --tag all12 --attack-cap 9312
tran

## 6. Zip only the new outputs (no parquet, nothing from `main`)

In [9]:
import zipfile
NAME = "nids_sweep"
dest = WORK.parent / f"{NAME}.zip"
pick = [p for p in WORK.rglob("*") if p.is_file() and p.suffix != ".parquet" and (
    any(part.startswith(("sweep_", "repro_s42", "all12_")) for part in p.relative_to(WORK).parts)
    or p.relative_to(WORK).parts[0] in ("sweep", "all12") or p.name == "all12_schedule.json")]
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in pick:
        zf.write(p, Path("nids_out") / p.relative_to(WORK))
print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
print("Save Version, then download", dest.name, "from the Output tab.")

855 files -> /kaggle/working/nids_sweep.zip (83.8 MB)
Save Version, then download nids_sweep.zip from the Output tab.
